In [ ]:
import requests
from json import loads
import pandas

from datetime import datetime

In [3]:
import os

# Access the API key from the environment variable
api_key = os.getenv('API_KEY')

if api_key is None:
    raise ValueError("API_KEY environment variable is not set!")

In [239]:
import string

def strip_punctuation(title):
    # Create a translation table that maps punctuation to None
    translator = str.maketrans('', '', string.punctuation)
    # Use the translation table to remove punctuation
    return title.translate(translator)

# Example usage:
original_title = "The Shawshank Redemption: A Classic?"
stripped_title = strip_punctuation(original_title)

print("Original Title:", original_title)
print("Stripped Title:", stripped_title)

Original Title: The Shawshank Redemption: A Classic?
Stripped Title: The Shawshank Redemption A Classic


In [195]:
import requests
from json import loads

def tmdb_api_top_rated():
    # API endpoint
    url = 'https://api.themoviedb.org/3/discover/movie'

    # Initialize an empty list to store movie titles
    movie_titles = []

    for i in range(1, 16):
        # Request parameters
        params = {
            "api_key": api_key,
            "language": "en-US",
            "release_date.gte": '2000-01-01',
            "release_date.lte": '2024-12-31',
            "sort_by": "vote_average.desc",
            "vote_count.gte": "300",
            "page": i
        }

        # Sending the GET request to the TMDB API
        response = requests.get(url, params=params)
        data = response.json()
        # Extract movie titles from the current page
        for movie in data['results']:
            movie_titles.append({
                'title': movie['title'],
                'vote_average': round(movie['vote_average'], 1),
                'Description': movie['overview']
            })

    return movie_titles

# Test the function
top_rated_movies = tmdb_api_top_rated()
print(top_rated_movies[0])


{'title': 'The Shawshank Redemption', 'vote_average': 8.7, 'Description': 'Imprisoned in the 1940s for the double murder of his wife and her lover, upstanding banker Andy Dufresne begins a new life at the Shawshank prison, where he puts his accounting skills to work for an amoral warden. During his long stretch in prison, Dufresne comes to be admired by the other inmates -- including an older prisoner named Red -- for his integrity and unquenchable sense of hope.'}


In [95]:
import requests
import json
from bs4 import BeautifulSoup
import re

def getImdb_movie_page(movie_title):

    search_url = f"https://www.imdb.com/search/title/?title={movie_title}" # search url to use for movie titles
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    movie_title = strip_punctuation(movie_title)
    
    response = requests.get(search_url, headers=headers)
    # Step 2: Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find all <a> tags that contain the title in the <h3> tag within them
    a_tags = soup.find_all('a', class_='ipc-title-link-wrapper')
   
    # Step 4: Extract and clean up the text from each found <a> element
    titles = []
    movie_urls = []
    for a_tag in a_tags:
        h3_tag = a_tag.find('h3', class_='ipc-title__text')  # Find the <h3> tag inside <a>
        if h3_tag:
            title = h3_tag.text.strip()  # Clean up title by removing extra spaces
            title = re.sub(r'^\d+\.\s?', '', title) # Remove any leading numbers (e.g., "1. ", "2. ")
            title = strip_punctuation(title)
            if title == movie_title:
                titles.append(title)
                movie_url = "https://www.imdb.com" + a_tag['href']  # Construct full URL
                movie_urls.append(movie_url)
                break

    return titles, movie_urls


movie_text = getImdb_movie_page('Spider-Man: No Way Home')
print(movie_text)


(['SpiderMan No Way Home'], ['https://www.imdb.com/title/tt10872600/?ref_=sr_t_1'])


In [262]:
import requests
import requests_cache
from bs4 import BeautifulSoup
import pandas as pd

# Enable cache
requests_cache.install_cache('imdb_cache', expire_after=3600)  # Cache expires after 1 hour

def get_imdb_info(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 6.0) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/70.0.3538.110 Safari/537.36',
    }
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Movie title
    movie_title = soup.find('span', class_='hero__primary-text')
    movie_title = movie_title.text if movie_title else None

    # IMDB rating
    imdb_rating_tag = soup.find('div', attrs={'data-testid': 'hero-rating-bar__aggregate-rating__score'})
    movie_imdb_rating = float(imdb_rating_tag.span.text) if imdb_rating_tag else None

    # Movie budget
    budget_tag = soup.find('span', string='Budget')
    next_span = budget_tag.find_next('span') if budget_tag else None
    movie_budget = next_span.text if next_span else None
    if movie_budget != None:
        movie_budget = re.sub(r'[^\d]', '', movie_budget)  # Remove non-digit characters (e.g., $, ,)
        movie_budget = float(movie_budget)  # Convert the string to a float
    else:
        movie_budget = None  # If not available

    # Movie Gross Worldwide
    gworld_tag = soup.find('span', string='Gross worldwide')
    next_span = gworld_tag.find_next('span') if gworld_tag else None
    movie_gworld = next_span.text if next_span else None
    if movie_gworld != None:
        movie_gworld = re.sub(r'[^\d.]', '', movie_gworld)
        movie_gworld = float(movie_gworld)
    else:
        movie_gworld = None

    # Movie Runtime
    runtime_tag = soup.find('li', attrs={'data-testid': 'title-techspec_runtime'})
    movie_runtime = runtime_tag.div.text if runtime_tag else None
    if movie_runtime != None:
        runtime_parts = movie_runtime.replace('hours', '').replace('hour', '').replace('minutes', '').replace('minute', '').strip()
        parts = runtime_parts.split()
        
        if len(parts) == 2:  # If both hours and minutes are present
            hours = int(parts[0])
            minutes = int(parts[1])
        elif len(parts) == 1:  # If only minutes are present
            hours = 0
            minutes = int(parts[0])
    
        # Calculate total runtime in minutes
        total_minutes = hours * 60 + minutes
        movie_runtime = total_minutes
    else:
        total_minutes = None

    # Country of origin
    ctry_origin_tag = soup.find('li', attrs={'data-testid': 'title-details-origin'})
    if ctry_origin_tag:
        country_links = ctry_origin_tag.find_all('a')  # Find all <a> tags within the section
        movie_ctry_origin = ', '.join([link.text.strip() for link in country_links])  # Join country names with a comma
    else:
        movie_ctry_origin = None

    # Movie Release Date
    rd_tag = soup.find('li', attrs={'data-testid': 'title-details-releasedate'})
    movie_rd = rd_tag.div.ul.li.a.text if rd_tag else None
    if movie_rd != None:
        movie_rd = movie_rd.split('(')[0].strip()
    else:
        movie_rd = None
 

    # Create a DataFrame
    return {
        'Title': strip_punctuation(movie_title),
        'IMDB rating': movie_imdb_rating,
        'Budget': movie_budget,
        'Gross worldwide': movie_gworld,
        'Runtime': total_minutes,
        'Country of Origin': movie_ctry_origin,
        'Release Date': movie_rd
    }

#movie_test = get_imdb_info('https://www.imdb.com/title/tt0468569/?ref_=nv_sr_srsg_0_tt_8_nm_0_in_0_q_the%2520dark%2520')
movie_test = get_imdb_info('https://www.imdb.com/title/tt0111161/?ref_=chttp_t_1')
movie_test2 = get_imdb_info('https://www.imdb.com/title/tt28035641/')
print(movie_test)
print(movie_test2)


{'Title': 'The Shawshank Redemption', 'IMDB rating': 9.3, 'Budget': 25000000.0, 'Gross worldwide': 29332133.0, 'Runtime': 142, 'Country of Origin': 'United States', 'Release Date': 'October 14, 1994'}
{'Title': 'Once Upon a Studio', 'IMDB rating': 8.3, 'Budget': None, 'Gross worldwide': None, 'Runtime': 9, 'Country of Origin': 'United States, Canada, Thailand', 'Release Date': 'October 13, 2023'}


In [259]:
import requests
import lxml.html as lx
import re
import time
import nltk
import numpy as np
from bs4 import BeautifulSoup

import matplotlib.pyplot as plt # or your favorite alternative

from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

def get_rotten_tomatoes_details(rt_url):
    """Scrapes details from a Rotten Tomatoes movie page."""
    response = requests.get(rt_url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')

    # Extract Rotten Tomatoes score
    rt_score_tag = soup.find('rt-text', attrs={'slot': 'criticsScore'})
    rt_score = rt_score_tag.text.strip() if rt_score_tag else None
    if rt_score:
        rt_score = float(rt_score.strip('%')) / 10
    else:
        rt_score = None

    # Extract Popcornmeter score
    pop_score_tag = soup.find('rt-text', attrs={'slot': 'audienceScore'})
    pop_score = pop_score_tag.text.strip() if pop_score_tag else None
    if pop_score:
        pop_score = float(pop_score.strip('%')) / 10
    else:
        pop_score = None

    item_values = soup.find_all('div', attrs={'class': 'category-wrap'})

    # Initialize values to avoid undefined variables
    first_production_co = None
    movie_rating = None
    genre_list = []

    # Finding Production Co, Rating, and Genre
    for tag in item_values:
        item_label = tag.find('rt-text', attrs={'data-qa': 'item-label'})
        if item_label:
            label_text = item_label.text.strip()

            if label_text == 'Director':
                next_rt_link = item_label.find_next('rt-link')
                director = next_rt_link.text.strip() if next_rt_link else None

            elif label_text == 'Production Co':
                next_rt_text = item_label.find_next('rt-text')
                first_production_co = next_rt_text.text.strip() if next_rt_text else None

            elif label_text == 'Rating':
                next_rt_text = item_label.find_next('rt-text')
                rating_text = next_rt_text.text.strip() if next_rt_text else None
                if rating_text:
                    movie_rating = re.search(r'([A-Z]+[-]?\d*)', rating_text).group() if rating_text else None

            elif label_text == 'Genre':
                next_rt_links = tag.find_all('rt-link')  # Find all genres
                for genre_tag in next_rt_links:
                    genre = genre_tag.text.strip()
                    if genre:
                        genre_list.append(genre)

    # If genre_list has any entries, join them with commas
    genre_list = ', '.join(genre_list) if genre_list else None
    
    return {'rt_score': rt_score,
            'pop_score': pop_score,
            'Director': director,
            'Production Co': first_production_co,
            'Movie Rating': movie_rating,
            'Genres': genre_list
    }


print(get_rotten_tomatoes_details('https://www.rottentomatoes.com/m/o_auto_da_compadecida'))

{'rt_score': None, 'pop_score': 9.5, 'Director': 'Guel Arraes', 'Production Co': 'Globo Filmes', 'Movie Rating': None, 'Genres': 'Fantasy, Comedy, Adventure'}


In [66]:

def getTomatometer(movie_name):

    search_url = f"https://www.rottentomatoes.com/search?search={movie_name}" # search url to use for movie titles
    response = requests.get(search_url)
    html = lx.fromstring(response.content)

    movie_rows = html.xpath('//*[@id="search-results"]/search-page-result[1]/ul/search-page-media-row') # look through movie section

    # Iterate through each movie row found and extract the title
    for row in movie_rows:
        movie = row.xpath('./a[2]/text()')
        movie = movie[0].strip()
        movie = strip_punctuation(movie)
        if movie != movie_name: # if not the same movie title, skip
            continue
        movie_link = row.xpath('./a[2]/@href')  # Get the movie link for the current row
        response = get_rotten_tomatoes_details(movie_link[0])  # Pass the movie link to the function
        return response
    return None

movie_text = getTomatometer('The Shawshank Redemption')
print(movie_text)

{'rt_score': '89%', 'rt_url': 'https://www.rottentomatoes.com/m/shawshank_redemption'}


In [253]:


def fetching_movies():

    movies_data = []
    
    # all_movies = tmdb_api()
    all_movies = tmdb_api_top_rated()

    for movie in all_movies:
        
        movie_title = movie['title']
        
        movie_details = getImdb_movie_page(movie_title)
        print(movie_details)
        
        if movie_details[1]:
            imdb_details = get_imdb_info(movie_details[1][0])
        
            # Fetch Rotten Tomatoes details
            rt_details = getTomatometer(imdb_details['Title'])
        
            # Ensure both dictionaries are not None
            imdb_details = imdb_details or {}  # If None, set to an empty dictionary
            rt_details = rt_details or {}      # If None, set to an empty dictionary
            
            # Combine data from IMDb and Rotten Tomatoes
            movie_details = {**imdb_details, **rt_details, 'tmdb_rating': movie['vote_average'], 'Description': movie['Description']}
            movies_data.append(movie_details)
        else:
            print("Not found")

    return movies_data
    

In [263]:
start_time = time.time()

top_movies = fetching_movies()

# Convert the data into a DataFrame
df = pd.DataFrame(top_movies)

# Save the DataFrame to a CSV file
csv_filename = 'fetching_popular_movies.csv'
df.to_csv(csv_filename, index=False)

end_time = time.time()
elapsed_time = end_time - start_time
print(f"\nTotal time taken: {elapsed_time:.2f} seconds")

(['The Shawshank Redemption'], ['https://www.imdb.com/title/tt0111161/?ref_=sr_t_1'])
(['The Godfather'], ['https://www.imdb.com/title/tt0068646/?ref_=sr_t_1'])
(['The Godfather Part II'], ['https://www.imdb.com/title/tt0071562/?ref_=sr_t_1'])
(['Schindlers List'], ['https://www.imdb.com/title/tt0108052/?ref_=sr_t_1'])
(['12 Angry Men'], ['https://www.imdb.com/title/tt0050083/?ref_=sr_t_1'])
(['Spirited Away'], ['https://www.imdb.com/title/tt0245429/?ref_=sr_t_1'])
(['Dilwale Dulhania Le Jayenge'], ['https://www.imdb.com/title/tt0112870/?ref_=sr_t_1'])
(['The Dark Knight'], ['https://www.imdb.com/title/tt0468569/?ref_=sr_t_1'])
(['Parasite'], ['https://www.imdb.com/title/tt6751668/?ref_=sr_t_1'])
(['The Green Mile'], ['https://www.imdb.com/title/tt0120689/?ref_=sr_t_1'])
(['Pulp Fiction'], ['https://www.imdb.com/title/tt0110912/?ref_=sr_t_1'])
(['Your Name'], ['https://www.imdb.com/title/tt5311514/?ref_=sr_t_2'])
(['The Lord of the Rings The Return of the King'], ['https://www.imdb.com